In [2]:
import pandas as pd

path = r"D:\multiview-log-anomaly-detection\data\raw\BGL.log"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    logs = f.readlines()

df = pd.DataFrame({"log": logs})

print(df.head())
print(df.shape)

                                                 log
0  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 20...
1  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 20...
2  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 20...
3  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 20...
4  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 20...
(4747963, 1)


In [3]:
df.columns

Index(['log'], dtype='object')

In [2]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/bgl_parsed.parquet")
emb = np.load("../data/processed/semantic_embeddings.npy")
struct = pd.read_parquet("../data/processed/structural_features.parquet")
temp = pd.read_parquet("../data/processed/temporal_features.parquet")

print("bgl_parsed rows:      ", len(df))
print("semantic emb rows:    ", emb.shape[0])
print("structural rows:      ", len(struct))
print("temporal rows:        ", len(temp))
print("emb has NaNs:         ", np.isnan(emb).any())

bgl_parsed rows:       4713483
semantic emb rows:     4713483
structural rows:       4713483
temporal rows:         4713483
emb has NaNs:          False


In [2]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/bgl_parsed.parquet")
sem = np.load("../data/processed/semantic_anomaly_scores.npy")
struct = np.load("../data/processed/structural_anomaly_scores.npy")
temp = np.load("../data/processed/temporal_anomaly_scores.npy")

print("bgl_parsed rows:  ", len(df))
print("semantic scores:  ", sem.shape[0])
print("structural scores:", struct.shape[0])
print("temporal scores:  ", temp.shape[0])
print("any NaN — sem/struct/temp:", np.isnan(sem).any(), np.isnan(struct).any(), np.isnan(temp).any())

bgl_parsed rows:   4713483
semantic scores:   4713483
structural scores: 4713483
temporal scores:   4713483
any NaN — sem/struct/temp: False False False


In [3]:
sample_mask = df["content"] == "instruction cache parity error corrected"
sample_idx = df[sample_mask].head(6).index

print("Semantic scores:  ", sem[sample_idx])
print("Structural scores:", struct[sample_idx])
print("Temporal scores:  ", temp[sample_idx])

Semantic scores:   [-2.3841858e-07 -2.3841858e-07 -2.3841858e-07 -2.3841858e-07
 -2.3841858e-07 -2.3841858e-07]
Structural scores: [0.16647429 0.16647429 0.16647429 0.16647429 0.16647429 0.16647429]
Temporal scores:   [0.         0.05101799 0.07215033 0.08383111 0.09126376 0.09641494]


In [4]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/bgl_parsed.parquet")
final_scores = np.load("../data/processed/final_anomaly_scores.npy")
weights = np.load("../data/processed/fusion_weights.npy")

sample_mask = df["content"] == "instruction cache parity error corrected"
sample_idx = df[sample_mask].head(6).index

print("Final fused scores:", final_scores[sample_idx])
print("\nWeights (sem, struct, temp) per row:")
print(weights[sample_idx])

Final fused scores: [0.05731564 0.08262676 0.09087223 0.0948549  0.09708309 0.09847279]

Weights (sem, struct, temp) per row:
[[0.23370225 0.34429155 0.4220062 ]
 [0.26545778 0.39107399 0.34346823]
 [0.28023415 0.41284263 0.30692322]
 [0.29071876 0.42828861 0.28099263]
 [0.29786795 0.43882084 0.26331121]
 [0.3027724  0.44604611 0.25118148]]


In [5]:
print("Row count check:", len(df) == final_scores.shape[0] == weights.shape[0])
print("Any NaN in final_scores:", np.isnan(final_scores).any())
print("Any NaN in weights:", np.isnan(weights).any())
print("Weights sum to 1 per row (spot check):", weights[:5].sum(axis=1))

Row count check: True
Any NaN in final_scores: False
Any NaN in weights: False
Weights sum to 1 per row (spot check): [1. 1. 1. 1. 1.]


In [6]:
is_anomaly = np.load("../data/processed/is_anomaly.npy")
severity_bucket = np.load("../data/processed/severity_bucket.npy")

sample_mask = df["content"] == "instruction cache parity error corrected"
sample_idx = df[sample_mask].head(6).index

print("Is anomaly:", is_anomaly[sample_idx])
print("Severity:  ", severity_bucket[sample_idx])

Is anomaly: [False False False False False False]
Severity:   ['NONE' 'NONE' 'NONE' 'NONE' 'NONE' 'NONE']


In [8]:
import json
with open("../data/processed/evidence_sample.json") as f:
    packages = json.load(f)

print(json.dumps(packages[1], indent=2, default=str))
print(json.dumps(packages[5], indent=2, default=str))

{
  "row_index": 3561348,
  "time": "2005-10-09 09:52:47.549495",
  "component": "HARDWARE",
  "node": "R27-M1",
  "content": "EndServiceAction is restarting the NodeCards in midplane R27-M1 as part of Service Action 473",
  "template": "EndServiceAction is restarting the <*> in midplane <*> as part of Service Action <*>",
  "final_anomaly_score": 0.4678252720170171,
  "severity": "MEDIUM",
  "per_view_deviation": {
    "semantic": {
      "score": 0.5989347696304321,
      "weight": 0.35283760479451015
    },
    "structural": {
      "score": 0.45008541942809843,
      "weight": 0.3889087130408644
    },
    "temporal": {
      "score": 0.31541242921097795,
      "weight": 0.2582536821646254
    }
  },
  "dominant_contributing_view": "semantic",
  "nearest_normal_example": [
    {
      "row_index": 836936,
      "time": "2005-06-17 08:32:22.689953",
      "content": "EndServiceAction is restarting the NodeCards in midplane R11-M1 as part of Service Action 221",
      "similarity": 1

In [10]:
import pandas as pd
sig_df = pd.read_parquet("../data/processed/incident_signatures.parquet")
top10 = sig_df.sort_values("n_anomalies", ascending=False).head(10)
for _, row in top10.iterrows():
    print(f"\nIncident {row['incident_id']} | n_anomalies={row['n_anomalies']} | n_distinct_templates={row['n_distinct_templates']}")
    print(f"  components: {row['components_involved']}")
    print(f"  duration: {row['start_time']} to {row['end_time']}")


Incident 3175 | n_anomalies=9549 | n_distinct_templates=6
  components: ['KERNEL']
  duration: 2005-12-01 08:29:05.695756 to 2005-12-01 08:47:49.895138

Incident 3173 | n_anomalies=9519 | n_distinct_templates=3
  components: ['KERNEL']
  duration: 2005-12-01 06:34:47.721810 to 2005-12-01 06:53:29.175537

Incident 951 | n_anomalies=5701 | n_distinct_templates=8
  components: ['KERNEL']
  duration: 2005-07-17 03:57:10.774076 to 2005-07-17 04:30:10.614800

Incident 1084 | n_anomalies=5600 | n_distinct_templates=8
  components: ['FATAL' 'KERNEL' 'LINKCARD']
  duration: 2005-07-23 16:58:52.008598 to 2005-07-23 18:14:37.819934

Incident 1181 | n_anomalies=5547 | n_distinct_templates=7
  components: ['FATAL' 'KERNEL' 'LINKCARD']
  duration: 2005-07-28 11:02:14.110053 to 2005-07-28 13:14:04.656252

Incident 200 | n_anomalies=4780 | n_distinct_templates=14
  components: ['KERNEL' 'LINKCARD' 'MMCS']
  duration: 2005-06-11 17:18:51.385127 to 2005-06-11 23:55:56.021490

Incident 2715 | n_anomalie

In [11]:
rank_df = pd.read_parquet("../data/processed/root_cause_rankings.parquet")
print(rank_df[rank_df["incident_id"] == 2926].sort_values("rank"))

      component           first_occurrence  in_cluster_freq  avg_severity  \
3548     KERNEL 2005-11-15 07:43:32.811651               20      0.520579   
3549   HARDWARE 2005-11-15 08:07:54.684183               53      0.454256   
3550  DISCOVERY 2005-11-15 08:16:27.846328               16      0.419570   
3551  BGLMASTER 2005-11-15 07:57:00.541566                1      0.381396   
3552       CMCS 2005-11-15 08:06:28.636528                3      0.374469   
3553       MMCS 2005-11-15 07:57:01.178203                3      0.352545   

      first_occurrence_priority  cooc_centrality  \
3548                   1.000000              614   
3549                   0.000684              244   
3550                   0.000506              218   
3551                   0.001237              145   
3552                   0.000726              115   
3553                   0.001236              185   

      first_occurrence_priority_norm  avg_severity_norm  in_cluster_freq_norm  \
3548          

In [12]:
import numpy as np, pandas as pd, json
from sklearn.metrics import roc_auc_score

df = pd.read_parquet("../data/processed/bgl_parsed.parquet")
sem = np.load("../data/processed/semantic_anomaly_scores.npy")
struct = np.load("../data/processed/structural_anomaly_scores.npy")
temp = np.load("../data/processed/temporal_anomaly_scores.npy")
final_scores = np.load("../data/processed/final_anomaly_scores.npy")

with open("../data/processed/split_indices.json") as f:
    split = json.load(f)
test_start = split["val_end_idx"]

y_true = (df["label"].values[test_start:] != "-").astype(int)

for name, scores in [("semantic", sem), ("structural", struct), ("temporal", temp), ("final_fused", final_scores)]:
    auc = roc_auc_score(y_true, scores[test_start:])
    print(f"{name}: AUC-ROC = {auc:.4f}")

semantic: AUC-ROC = 0.3059
structural: AUC-ROC = 0.5467
temporal: AUC-ROC = 0.4277
final_fused: AUC-ROC = 0.4561


In [2]:
import numpy as np
severity_bucket = np.load("../data/processed/severity_bucket.npy")
print(np.unique(severity_bucket, return_counts=True))

(array(['CRITICAL', 'LOW', 'MEDIUM', 'NONE'], dtype='<U8'), array([  10753,  191327,  335558, 4175845]))
